<a href="https://colab.research.google.com/github/nohaelkachach/casiav2-splicing-gradcam-audit/blob/main/05_regression_analysis_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Statistical analysis (rerun on leak-free Grad-CAM data from notebook 04)

Tests the central hypothesis: does splice size moderate the effect of contrast polarity on Grad-CAM's localization faithfulness (IoU against ground-truth masks)?

**This notebook documents the full specification search honestly, in the order it was actually conducted:**
1. Continuous specifications (linear, log-transformed, quadratic) are tested first.
2. The linear specification's leverage/instability is diagnosed directly (Cook's distance vs. distance from the predictor's mean).
3. Robust regression (Huber, Tukey biweight) is tested as a continuous alternative that down-weights high-leverage points.
4. A threshold-based specification is fit and validated via a distribution-free test (Mann-Whitney U), raw group means, a second outcome measure (AUC-IoU), and replication on a second architecture (EfficientNet-B0).

**Every cell below reloads its inputs from disk rather than relying on variables set earlier in the session** — this avoids the exact staleness bug from the previous run, where a cell's cached output looked unchanged after the underlying CSV had actually been updated. If you rerun this notebook after regenerating the CSVs in notebook 04, run every cell fresh, top to bottom, rather than re-executing individual cells out of order.

## Setup — load ResNet18 regression-ready data, with a freshness check

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.robust.norms as robust_norms
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import mannwhitneyu

RESNET_CSV = "/content/drive/MyDrive/CASIA2.0/regression_ready_resnet.csv"
EFFNET_CSV = "/content/drive/MyDrive/CASIA2.0/regression_ready_effnet.csv"

resnet_df = pd.read_csv(RESNET_CSV)
resnet_df["polarity_bin"] = (resnet_df["polarity"] == "dark_on_bright").astype(int)

mtime = time.ctime(os.path.getmtime(RESNET_CSV))
print(f"Loaded {RESNET_CSV}")
print(f"File last modified: {mtime}")
print(f"n = {len(resnet_df)}")
print(f"Mean IoU: {resnet_df['iou'].mean():.4f}  (sanity check against notebook 04's printed .describe())")
print(resnet_df["polarity_bin"].value_counts())

Mounted at /content/drive
Loaded /content/drive/MyDrive/CASIA2.0/regression_ready_resnet.csv
File last modified: Fri Aug 28 01:03:37 2026
n = 1828
Mean IoU: 0.1150  (sanity check against notebook 04's printed .describe())
polarity_bin
0    934
1    894
Name: count, dtype: int64


## Step 1 — Continuous specifications (tested first, in this order)

In [2]:
# --- Linear ---
model_linear = "iou ~ polarity_bin * splice_size_frac + abs_contrast"
ols_linear = smf.ols(formula=model_linear, data=resnet_df).fit(cov_type="HC3")
print("=== Linear (HC3) ===")
print(ols_linear.summary().tables[1])

# --- Log-transformed ---
resnet_df["log_splice_size"] = np.log(resnet_df["splice_size_frac"] + 0.001)
model_log = "iou ~ polarity_bin * log_splice_size + abs_contrast"
ols_log = smf.ols(formula=model_log, data=resnet_df).fit(cov_type="HC3")
print("\n=== Log-transformed (HC3) ===")
print(ols_log.summary().tables[1])

# --- Quadratic ---
resnet_df["splice_size_sq"] = resnet_df["splice_size_frac"] ** 2
model_quad = "iou ~ polarity_bin * splice_size_frac + polarity_bin * splice_size_sq + abs_contrast"
ols_quad = smf.ols(formula=model_quad, data=resnet_df).fit(cov_type="HC3")
print("\n=== Quadratic (HC3) ===")
print(ols_quad.summary().tables[1])

=== Linear (HC3) ===
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept                         0.0876      0.005     17.487      0.000       0.078       0.097
polarity_bin                      0.0074      0.005      1.351      0.177      -0.003       0.018
splice_size_frac                  0.2264      0.019     11.976      0.000       0.189       0.263
polarity_bin:splice_size_frac     0.1569      0.049      3.210      0.001       0.061       0.253
abs_contrast                     -0.0003   7.65e-05     -3.987      0.000      -0.000      -0.000

=== Log-transformed (HC3) ===
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept                        0.2179      0.009     24.644      0

## Step 2 — Diagnose the linear specification's instability

In [3]:
influence = ols_linear.get_influence()
cooks_d = influence.cooks_distance[0]
hat_values = influence.hat_matrix_diag

threshold = 4 / len(resnet_df)
n_influential = (cooks_d > threshold).sum()
print(f"Influential points (Cook's D > 4/n): {n_influential} of {len(resnet_df)}")

resnet_df["leverage"] = hat_values
resnet_df["dist_from_mean_size"] = np.abs(resnet_df["splice_size_frac"] - resnet_df["splice_size_frac"].mean())
corr_leverage = resnet_df["leverage"].corr(resnet_df["dist_from_mean_size"])
print(f"Correlation between leverage and distance from mean splice size: {corr_leverage:.3f}")

influential_df = resnet_df[cooks_d > threshold]
normal_df = resnet_df[cooks_d <= threshold]
print(f"\nInfluential points — mean splice_size_frac: {influential_df['splice_size_frac'].mean():.3f}")
print(f"Non-influential points — mean splice_size_frac: {normal_df['splice_size_frac'].mean():.3f}")

Influential points (Cook's D > 4/n): 126 of 1828
Correlation between leverage and distance from mean splice size: 0.752

Influential points — mean splice_size_frac: 0.400
Non-influential points — mean splice_size_frac: 0.113


## Step 3 — Robust regression on continuous splice size (Huber / Tukey biweight M-estimation)
Tests whether down-weighting — rather than binning — the high-leverage large-splice points recovers a stable continuous interaction.

**Note:** `statsmodels.RLM` implements M-estimation (IRLS from the OLS start), not a true MM-estimator (Yohai, 1987), which would require a high-breakdown initial fit not available in a standard Python stack. Report this as M-estimation, not MM-estimation.

In [4]:
resnet_df["interaction"] = resnet_df["polarity_bin"] * resnet_df["splice_size_frac"]
X_robust = resnet_df[["polarity_bin", "splice_size_frac", "interaction", "abs_contrast"]].copy()
X_robust = sm.add_constant(X_robust)
y_robust = resnet_df["iou"]

huber_fit = sm.RLM(y_robust, X_robust, M=robust_norms.HuberT()).fit()
print("=== Robust regression: Huber's T M-estimator ===")
print(huber_fit.summary())

tukey_fit = sm.RLM(y_robust, X_robust, M=robust_norms.TukeyBiweight()).fit()
print("\n=== Robust regression: Tukey biweight M-estimator ===")
print(tukey_fit.summary())

resnet_df["huber_weight"] = huber_fit.weights
resnet_df["tukey_weight"] = tukey_fit.weights
influential_mask = cooks_d > threshold
print("\n--- Mean estimator weight: high-Cook's-D points vs. rest ---")
print(f"Huber  - influential: {resnet_df.loc[influential_mask, 'huber_weight'].mean():.3f}  "
      f"| rest: {resnet_df.loc[~influential_mask, 'huber_weight'].mean():.3f}")
print(f"Tukey  - influential: {resnet_df.loc[influential_mask, 'tukey_weight'].mean():.3f}  "
      f"| rest: {resnet_df.loc[~influential_mask, 'tukey_weight'].mean():.3f}")

print("\n--- Interaction term ---")
print(f"Huber:  coef={huber_fit.params['interaction']:.4f}, p={huber_fit.pvalues['interaction']:.4f}")
print(f"Tukey:  coef={tukey_fit.params['interaction']:.4f}, p={tukey_fit.pvalues['interaction']:.4f}")

=== Robust regression: Huber's T M-estimator ===
                    Robust linear Model Regression Results                    
Dep. Variable:                    iou   No. Observations:                 1828
Model:                            RLM   Df Residuals:                     1823
Method:                          IRLS   Df Model:                            4
Norm:                          HuberT                                         
Scale Est.:                       mad                                         
Cov Type:                          H1                                         
Date:                Fri, 28 Aug 2026                                         
Time:                        01:15:32                                         
No. Iterations:                    27                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------

## Step 4 — Threshold-based specification (primary result)
Splice size dichotomized at two cutoffs — sample median and top-33% quantile.

In [5]:
median_size = resnet_df["splice_size_frac"].median()
resnet_df["large_splice_median"] = (resnet_df["splice_size_frac"] >= median_size).astype(int)
model_median = "iou ~ polarity_bin * large_splice_median + abs_contrast"
ols_median = smf.ols(formula=model_median, data=resnet_df).fit(cov_type="HC3")
print(f"=== Median split (cutoff={median_size:.4f}) ===")
print(ols_median.summary().tables[1])

cutoff_33 = resnet_df["splice_size_frac"].quantile(0.67)
resnet_df["large_splice_top33"] = (resnet_df["splice_size_frac"] >= cutoff_33).astype(int)
model_top33 = "iou ~ polarity_bin * large_splice_top33 + abs_contrast"
ols_top33 = smf.ols(formula=model_top33, data=resnet_df).fit(cov_type="HC3")
print(f"\n=== Top-33% split (cutoff={cutoff_33:.4f}) ===")
print(ols_top33.summary().tables[1])

=== Median split (cutoff=0.0621) ===
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                            0.0653      0.004     15.890      0.000       0.057       0.073
polarity_bin                         0.0005      0.004      0.147      0.883      -0.007       0.008
large_splice_median                  0.0993      0.006     15.506      0.000       0.087       0.112
polarity_bin:large_splice_median     0.0443      0.010      4.485      0.000       0.025       0.064
abs_contrast                        -0.0002   7.05e-05     -3.015      0.003      -0.000   -7.44e-05

=== Top-33% split (cutoff=0.1211) ===
                                      coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept        

## Step 5 — Distribution-free validation (Mann-Whitney U)
`stratified_mannwhitney` prints n, p, and r at each cutoff. Direction check printed explicitly — states which polarity group has the higher mean at each cutoff, so a sign flip (like the one caught in the previous run) is visible immediately rather than requiring separate raw-means inspection.

In [6]:
def stratified_mannwhitney(df, outcome="iou"):
    for pct in [0.40, 0.33, 0.30, 0.25, 0.20]:
        cutoff = df["splice_size_frac"].quantile(1 - pct)
        large = df[df["splice_size_frac"] >= cutoff]
        dark = large[large["polarity_bin"] == 1][outcome]
        bright = large[large["polarity_bin"] == 0][outcome]
        stat, p = mannwhitneyu(dark, bright, alternative="two-sided")
        r = 1 - (2 * stat) / (len(dark) * len(bright))
        higher = "dark_on_bright" if dark.mean() > bright.mean() else "bright_on_dark"
        print(f"Top {int(pct*100)}%: n_dark={len(dark)}, n_bright={len(bright)}, p={p:.4f}, r={r:.3f} "
              f"| mean(dark)={dark.mean():.4f}, mean(bright)={bright.mean():.4f}, higher={higher}")

print("=== ResNet18, outcome=iou ===")
stratified_mannwhitney(resnet_df, outcome="iou")

=== ResNet18, outcome=iou ===
Top 40%: n_dark=278, n_bright=453, p=0.0000, r=-0.212 | mean(dark)=0.2142, mean(bright)=0.1593, higher=dark_on_bright
Top 33%: n_dark=214, n_bright=389, p=0.0000, r=-0.202 | mean(dark)=0.2149, mean(bright)=0.1571, higher=dark_on_bright
Top 30%: n_dark=181, n_bright=368, p=0.0005, r=-0.183 | mean(dark)=0.2108, mean(bright)=0.1545, higher=dark_on_bright
Top 25%: n_dark=136, n_bright=321, p=0.0167, r=-0.141 | mean(dark)=0.2054, mean(bright)=0.1545, higher=dark_on_bright
Top 20%: n_dark=99, n_bright=267, p=0.0386, r=-0.141 | mean(dark)=0.2040, mean(bright)=0.1559, higher=dark_on_bright


## Step 6 — Raw group means (top-33% cutoff)
Recomputed directly from `resnet_df` in this cell — not read from any earlier cached output — to avoid the staleness issue from the previous run.

In [7]:
large = resnet_df[resnet_df["large_splice_top33"] == 1]
means = large.groupby("polarity_bin")["iou"].agg(["mean", "std", "count"])
print(means)
print("\n(polarity_bin: 0 = bright_on_dark, 1 = dark_on_bright)")
higher = "dark_on_bright (1)" if means.loc[1, "mean"] > means.loc[0, "mean"] else "bright_on_dark (0)"
print(f"Higher mean IoU in this stratum: {higher}")

                  mean      std  count
polarity_bin                          
0             0.157116  0.14220    389
1             0.214856  0.16759    214

(polarity_bin: 0 = bright_on_dark, 1 = dark_on_bright)
Higher mean IoU in this stratum: dark_on_bright (1)


## Step 7 — Robustness check: AUC-IoU (threshold-free outcome)
Repeats the median-split regression and stratified Mann-Whitney using `auc_iou` instead of `iou`.

In [8]:
model_median_auc = "auc_iou ~ polarity_bin * large_splice_median + abs_contrast"
ols_median_auc = smf.ols(formula=model_median_auc, data=resnet_df).fit(cov_type="HC3")
print("=== ResNet18, median split, outcome=auc_iou ===")
print(ols_median_auc.summary().tables[1])

print("\n=== ResNet18, stratified Mann-Whitney, outcome=auc_iou ===")
stratified_mannwhitney(resnet_df, outcome="auc_iou")

=== ResNet18, median split, outcome=auc_iou ===
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                            0.0672      0.004     15.712      0.000       0.059       0.076
polarity_bin                         0.0030      0.005      0.666      0.506      -0.006       0.012
large_splice_median                  0.0245      0.005      4.815      0.000       0.014       0.034
polarity_bin:large_splice_median     0.0484      0.008      5.911      0.000       0.032       0.064
abs_contrast                        -0.0002   5.48e-05     -3.765      0.000      -0.000    -9.9e-05

=== ResNet18, stratified Mann-Whitney, outcome=auc_iou ===
Top 40%: n_dark=278, n_bright=453, p=0.0000, r=-0.260 | mean(dark)=0.1280, mean(bright)=0.0783, higher=dark_on_bright
Top 33%: n_dark=214, n_bright=389, p=0.0000, r=-0.223 | mean(dark)=

## Step 8 — Cross-architecture replication (EfficientNet-B0)
Loaded fresh from disk, same as ResNet18 above — not reused from any earlier session variable.

In [9]:
effnet_df = pd.read_csv(EFFNET_CSV)
effnet_df["polarity_bin"] = (effnet_df["polarity"] == "dark_on_bright").astype(int)

eff_mtime = time.ctime(os.path.getmtime(EFFNET_CSV))
print(f"Loaded {EFFNET_CSV}")
print(f"File last modified: {eff_mtime}")
print(f"n = {len(effnet_df)}, mean IoU = {effnet_df['iou'].mean():.4f}")

median_size_eff = effnet_df["splice_size_frac"].median()
effnet_df["large_splice_median"] = (effnet_df["splice_size_frac"] >= median_size_eff).astype(int)

model_median_eff = "iou ~ polarity_bin * large_splice_median + abs_contrast"
ols_median_eff = smf.ols(formula=model_median_eff, data=effnet_df).fit(cov_type="HC3")
print("\n=== EfficientNet-B0, median split, outcome=iou ===")
print(ols_median_eff.summary().tables[1])

model_median_eff_auc = "auc_iou ~ polarity_bin * large_splice_median + abs_contrast"
ols_median_eff_auc = smf.ols(formula=model_median_eff_auc, data=effnet_df).fit(cov_type="HC3")
print("\n=== EfficientNet-B0, median split, outcome=auc_iou ===")
print(ols_median_eff_auc.summary().tables[1])

print("\n=== EfficientNet-B0, stratified Mann-Whitney, outcome=iou ===")
stratified_mannwhitney(effnet_df, outcome="iou")

print("\n=== EfficientNet-B0, stratified Mann-Whitney, outcome=auc_iou ===")
stratified_mannwhitney(effnet_df, outcome="auc_iou")

Loaded /content/drive/MyDrive/CASIA2.0/regression_ready_effnet.csv
File last modified: Fri Aug 28 01:03:39 2026
n = 1828, mean IoU = 0.1167

=== EfficientNet-B0, median split, outcome=iou ===
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                           -0.0176      0.005     -3.324      0.001      -0.028      -0.007
polarity_bin                        -0.0045      0.002     -2.124      0.034      -0.009      -0.000
large_splice_median                  0.2848      0.011     24.842      0.000       0.262       0.307
polarity_bin:large_splice_median    -0.1649      0.015    -10.890      0.000      -0.195      -0.135
abs_contrast                         0.0006      0.000      5.439      0.000       0.000       0.001

=== EfficientNet-B0, median split, outcome=auc_iou ===
                                       coef  

## Summary
Fill in after running: populate coefficients/p-values/direction from the actual output above before writing the paper's Results section. Pay attention to the `higher=` field printed in Step 5/6 — confirm which polarity group actually has the higher IoU on this run before reusing any wording from a previous draft, since direction is not guaranteed to match earlier (leaked) results.